# 🏗️ Notebook 2: Building a Hash Ring from Scratch

## What You'll Learn

1. How hash functions map strings to numbers
2. How to build a `ConsistentHashRing` class step by step
3. How to **visualize** the ring and key distribution with matplotlib
4. How the number of virtual nodes affects load balance

> **Prerequisites:** Complete Notebook 1 first to understand the BAD → BETTER → BEST progression.

In [ ]:
import hashlib
import bisect
import math
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from collections import Counter, defaultdict

---

## Step 1: Understanding Hash Functions

A hash function takes any input and returns a fixed-size number.
For consistent hashing, we need a function that:

1. **Deterministic** — same input always gives the same output
2. **Uniform** — outputs are spread evenly across the number space
3. **Fast** — we'll call it thousands of times

We'll use **MD5** (128-bit output). It's not cryptographically secure for passwords,
but it's perfect for hash rings because it distributes values evenly.

In [ ]:
def hash_to_ring(key: str, ring_size: int = 2**32) -> int:
    """Hash a string to a position on the ring [0, ring_size)."""
    digest = hashlib.md5(key.encode()).hexdigest()
    return int(digest, 16) % ring_size

# Demo: hash a few keys and see where they land
RING_SIZE = 2**32
print(f"Ring size: 0 to {RING_SIZE - 1:,} ({RING_SIZE:,} positions)\n")

for key in ["Server-A", "Server-B", "Server-C", "user:42", "event:1234"]:
    pos = hash_to_ring(key)
    pct = pos / RING_SIZE * 100
    print(f"  '{key}' \u2192 position {pos:,}  ({pct:.2f}% around the ring)")

---

## Step 2: The Ring Concept

Imagine the numbers 0 to 2³²-1 arranged in a circle, like a clock:

```
              0
          ╭───────╮
        ╯           ╰
      ╯    2³²       ╰
     │   positions    │
      ╰               ╯
        ╰           ╯
          ╰───────╯
           2³¹
```

**Servers** are placed at positions determined by hashing their names.
**Keys** are placed at positions determined by hashing the key.
Each key is assigned to the **next server clockwise** on the ring.

---

## Step 3: Building the `ConsistentHashRing` Class

Let's build this piece by piece. Our class needs:

| Method | Purpose |
|--------|--------|
| `__init__` | Set up the ring with a configurable number of virtual nodes |
| `_hash` | Hash a string to a ring position |
| `add_node` | Add a server (with virtual nodes) to the ring |
| `remove_node` | Remove a server from the ring |
| `get_node` | Find which server owns a given key |
| `get_distribution` | Count keys per server for analysis |

In [ ]:
class ConsistentHashRing:
    """
    A consistent hash ring with virtual nodes.

    Each physical server gets `num_virtual_nodes` positions on the ring.
    Keys are assigned to the nearest server clockwise.
    """

    def __init__(self, num_virtual_nodes: int = 150):
        self.num_virtual_nodes = num_virtual_nodes
        self.ring = {}            # ring_position -> physical node name
        self.sorted_positions = [] # sorted list of occupied positions
        self.nodes = set()        # set of physical node names

    def _hash(self, key: str) -> int:
        """Hash a string to a position on the ring [0, 2^32)."""
        return int(hashlib.md5(key.encode()).hexdigest(), 16) % (2**32)

    def add_node(self, node: str):
        """Add a physical node with virtual nodes spread around the ring."""
        self.nodes.add(node)
        for i in range(self.num_virtual_nodes):
            # Each virtual node gets a unique label: "Server-A#vn0", "Server-A#vn1", etc.
            virtual_label = f"{node}#vn{i}"
            position = self._hash(virtual_label)
            self.ring[position] = node
            bisect.insort(self.sorted_positions, position)

    def remove_node(self, node: str):
        """Remove a physical node and all its virtual nodes from the ring."""
        self.nodes.discard(node)
        for i in range(self.num_virtual_nodes):
            virtual_label = f"{node}#vn{i}"
            position = self._hash(virtual_label)
            if position in self.ring:
                del self.ring[position]
                self.sorted_positions.remove(position)

    def get_node(self, key: str) -> str:
        """Find which physical node owns this key.

        We hash the key, then walk clockwise to find the first server.
        Uses binary search (bisect) for O(log n) performance.
        """
        if not self.ring:
            return None

        position = self._hash(key)
        # Binary search: find first ring position >= key position
        idx = bisect.bisect_right(self.sorted_positions, position)
        # Wrap around if we went past the end (the ring is circular)
        if idx == len(self.sorted_positions):
            idx = 0
        return self.ring[self.sorted_positions[idx]]

    def get_distribution(self, keys: list) -> dict:
        """Count how many keys each physical node owns."""
        counts = Counter()
        for key in keys:
            node = self.get_node(key)
            counts[node] += 1
        return dict(sorted(counts.items()))

print("✅ ConsistentHashRing class ready!")
print(f"   Methods: add_node, remove_node, get_node, get_distribution")

---

## Step 4: Using the Hash Ring

In [ ]:
# Create a ring with 150 virtual nodes per server
ring = ConsistentHashRing(num_virtual_nodes=150)

# Add 4 servers
servers = ["Server-A", "Server-B", "Server-C", "Server-D"]
for server in servers:
    ring.add_node(server)

print(f"Ring has {len(ring.sorted_positions)} positions ({len(servers)} servers × {ring.num_virtual_nodes} vnodes)\n")

# Look up a few keys
test_keys = ["user:1", "user:2", "order:100", "product:42", "session:xyz"]
for key in test_keys:
    node = ring.get_node(key)
    print(f"  '{key}' \u2192 {node}")

In [ ]:
# Distribute 10,000 keys and check balance
keys = [f"key:{i}" for i in range(10_000)]
dist = ring.get_distribution(keys)

print("📊 Key distribution across 4 servers (10,000 keys):\n")
for node, count in dist.items():
    bar = "█" * (count // 50)
    print(f"  {node}: {count:>5,} keys  {bar}")

values = list(dist.values())
ideal = 10_000 // len(dist)
std_dev = (sum((v - ideal)**2 for v in values) / len(values)) ** 0.5
print(f"\n  Ideal per server: {ideal:,}")
print(f"  Std deviation: {std_dev:.0f} keys")
print(f"  Max/Min ratio: {max(values)/min(values):.2f}x")

---

## Step 5: Visualizing the Hash Ring

Let's draw the ring! We'll show:
- **Server positions** as colored dots around the circle
- **Key positions** as small markers
- **Ownership arcs** showing which server "owns" which part of the ring

In [ ]:
def visualize_ring(ring: ConsistentHashRing, sample_keys: list = None, title: str = "Consistent Hash Ring"):
    """Draw the hash ring showing server positions and optional key positions."""
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.set_aspect('equal')
    ax.set_xlim(-1.4, 1.4)
    ax.set_ylim(-1.4, 1.4)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

    # Draw the ring circle
    circle = plt.Circle((0, 0), 1.0, fill=False, color='gray', linewidth=2, linestyle='--')
    ax.add_patch(circle)

    # Color map for servers
    colors = {'Server-A': '#e74c3c', 'Server-B': '#3498db',
              'Server-C': '#2ecc71', 'Server-D': '#f39c12',
              'Server-E': '#9b59b6'}
    max_ring = 2**32

    # Draw virtual node positions (small dots)
    for pos in ring.sorted_positions:
        angle = 2 * math.pi * pos / max_ring - math.pi / 2  # start from top
        x = math.cos(angle) * 1.0
        y = math.sin(angle) * 1.0
        node = ring.ring[pos]
        color = colors.get(node, 'gray')
        ax.plot(x, y, 'o', color=color, markersize=3, alpha=0.4)

    # Draw sample keys if provided
    if sample_keys:
        for key in sample_keys:
            pos = ring._hash(key)
            angle = 2 * math.pi * pos / max_ring - math.pi / 2
            x = math.cos(angle) * 0.92
            y = math.sin(angle) * 0.92
            ax.plot(x, y, 'x', color='black', markersize=4, alpha=0.3)

    # Legend
    handles = [mpatches.Patch(color=colors.get(n, 'gray'), label=n) for n in sorted(ring.nodes)]
    if sample_keys:
        handles.append(plt.Line2D([0], [0], marker='x', color='black', label='Keys',
                                   linestyle='None', markersize=6))
    ax.legend(handles=handles, loc='upper right', fontsize=10)

    # Mark 0 position
    ax.annotate('0', xy=(0, 1.12), ha='center', fontsize=10, color='gray')

    plt.tight_layout()
    plt.show()

# Visualize our ring with 200 sample keys
sample = [f"key:{i}" for i in range(200)]
visualize_ring(ring, sample_keys=sample, title="Hash Ring: 4 Servers × 150 Virtual Nodes")

---

## Step 6: Analyzing Key Distribution

Let's visualize how evenly keys are distributed across servers.

In [ ]:
def plot_distribution(ring: ConsistentHashRing, num_keys: int = 10_000, title: str = ""):
    """Bar chart of key distribution across servers."""
    keys = [f"key:{i}" for i in range(num_keys)]
    dist = ring.get_distribution(keys)

    colors_map = {'Server-A': '#e74c3c', 'Server-B': '#3498db',
                  'Server-C': '#2ecc71', 'Server-D': '#f39c12',
                  'Server-E': '#9b59b6'}

    fig, ax = plt.subplots(figsize=(8, 4))
    nodes = list(dist.keys())
    counts = list(dist.values())
    bar_colors = [colors_map.get(n, 'gray') for n in nodes]

    bars = ax.bar(nodes, counts, color=bar_colors, edgecolor='white', linewidth=1.5)
    ideal = num_keys / len(nodes)
    ax.axhline(y=ideal, color='black', linestyle='--', alpha=0.5, label=f'Ideal: {ideal:.0f}')

    ax.set_ylabel('Number of Keys')
    ax.set_title(title or f'Key Distribution ({num_keys:,} keys)', fontweight='bold')
    ax.legend()

    # Add count labels on bars
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                f'{count:,}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.show()

plot_distribution(ring, title="Key Distribution: 4 Servers × 150 Virtual Nodes")

---

## Step 7: How Virtual Node Count Affects Balance

More virtual nodes = more even distribution. But how many do we actually need?
Let's test with different counts and measure the standard deviation.

In [ ]:
def measure_balance(num_virtual_nodes: int, servers: list, num_keys: int = 10_000) -> float:
    """Return the std deviation of key counts across servers."""
    r = ConsistentHashRing(num_virtual_nodes=num_virtual_nodes)
    for s in servers:
        r.add_node(s)
    keys = [f"key:{i}" for i in range(num_keys)]
    dist = r.get_distribution(keys)
    values = list(dist.values())
    ideal = num_keys / len(values)
    return (sum((v - ideal)**2 for v in values) / len(values)) ** 0.5

servers = ["Server-A", "Server-B", "Server-C", "Server-D"]
vnode_counts = [1, 5, 10, 25, 50, 100, 150, 200, 300, 500]
std_devs = []

col1 = "Virtual Nodes"
col2 = "Std Dev"
col3 = "Balance Quality"
print(f"{col1:>15} {col2:>10} {col3:>20}")
print("-" * 50)
for vn in vnode_counts:
    sd = measure_balance(vn, servers)
    std_devs.append(sd)
    if sd < 50:
        quality = "✅ Excellent"
    elif sd < 150:
        quality = "🟡 Good"
    elif sd < 500:
        quality = "🟠 Okay"
    else:
        quality = "🔴 Poor"
    print(f"{vn:>15} {sd:>10.0f} {quality:>20}")

In [ ]:
# Plot the relationship
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(vnode_counts, std_devs, 'o-', color='#3498db', linewidth=2, markersize=8)
ax.axhline(y=50, color='green', linestyle='--', alpha=0.5, label='Excellent threshold')
ax.set_xlabel('Number of Virtual Nodes per Server')
ax.set_ylabel('Standard Deviation (lower = more balanced)')
ax.set_title('Balance vs. Virtual Node Count', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Sweet spot: 100–200 virtual nodes gives excellent balance")
print("   with reasonable memory usage.")

---

## Step 8: Simulating Node Changes

Let's visualize what happens when we add and remove servers.

In [ ]:
ring_sim = ConsistentHashRing(num_virtual_nodes=150)
for s in ["Server-A", "Server-B", "Server-C"]:
    ring_sim.add_node(s)

keys = [f"key:{i}" for i in range(10_000)]
before = {k: ring_sim.get_node(k) for k in keys}

# Add Server-D
ring_sim.add_node("Server-D")
after_add = {k: ring_sim.get_node(k) for k in keys}

moved = sum(1 for k in keys if before[k] != after_add[k])
pct = moved / len(keys) * 100

print(f"📥 Adding Server-D to a 3-server ring:")
print(f"   Keys moved: {moved:,} / {len(keys):,} ({pct:.1f}%)")
print(f"   Ideal: {100/4:.1f}% (each of 4 servers should take 1/4 of total)\n")

# Where did moved keys come from?
moved_from = Counter()
for k in keys:
    if before[k] != after_add[k]:
        moved_from[before[k]] += 1

print("   Keys donated by each existing server:")
for server, count in sorted(moved_from.items()):
    print(f"     {server}: {count:,} keys")
print(f"\n   ✅ Load was taken evenly from all existing servers!")

In [ ]:
# Remove Server-B
ring_sim2 = ConsistentHashRing(num_virtual_nodes=150)
for s in ["Server-A", "Server-B", "Server-C", "Server-D"]:
    ring_sim2.add_node(s)

before2 = {k: ring_sim2.get_node(k) for k in keys}
ring_sim2.remove_node("Server-B")
after_rem = {k: ring_sim2.get_node(k) for k in keys}

moved = sum(1 for k in keys if before2[k] != after_rem[k])
pct = moved / len(keys) * 100

print(f"📤 Removing Server-B from a 4-server ring:")
print(f"   Keys moved: {moved:,} / {len(keys):,} ({pct:.1f}%)\n")

# Where did Server-B's keys go?
moved_to = Counter()
for k in keys:
    if before2[k] != after_rem[k]:
        moved_to[after_rem[k]] += 1

print("   Server-B's keys redistributed to:")
for server, count in sorted(moved_to.items()):
    print(f"     {server}: {count:,} keys")
print(f"\n   ✅ Load spread evenly across remaining servers!")

In [ ]:
# Side-by-side distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

colors_map = {'Server-A': '#e74c3c', 'Server-B': '#3498db',
              'Server-C': '#2ecc71', 'Server-D': '#f39c12'}

# Before (3 servers)
r1 = ConsistentHashRing(150)
for s in ["Server-A", "Server-B", "Server-C"]:
    r1.add_node(s)
d1 = r1.get_distribution(keys)
nodes1 = list(d1.keys())
axes[0].bar(nodes1, [d1[n] for n in nodes1], color=[colors_map[n] for n in nodes1])
axes[0].set_title("Before: 3 Servers", fontweight='bold')
axes[0].set_ylim(0, 5000)
axes[0].axhline(y=10000/3, color='black', linestyle='--', alpha=0.3)

# After adding (4 servers)
r1.add_node("Server-D")
d2 = r1.get_distribution(keys)
nodes2 = list(d2.keys())
axes[1].bar(nodes2, [d2[n] for n in nodes2], color=[colors_map[n] for n in nodes2])
axes[1].set_title("After Adding Server-D", fontweight='bold')
axes[1].set_ylim(0, 5000)
axes[1].axhline(y=10000/4, color='black', linestyle='--', alpha=0.3)

# After removing (3 servers, no B)
r3 = ConsistentHashRing(150)
for s in ["Server-A", "Server-B", "Server-C", "Server-D"]:
    r3.add_node(s)
r3.remove_node("Server-B")
d3 = r3.get_distribution(keys)
nodes3 = list(d3.keys())
axes[2].bar(nodes3, [d3[n] for n in nodes3], color=[colors_map[n] for n in nodes3])
axes[2].set_title("After Removing Server-B", fontweight='bold')
axes[2].set_ylim(0, 5000)
axes[2].axhline(y=10000/3, color='black', linestyle='--', alpha=0.3)

for ax in axes:
    ax.set_ylabel("Keys")

plt.suptitle("Key Distribution During Cluster Changes", fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🎯 Key Takeaways

1. **Hash functions** convert strings to ring positions — MD5/SHA give uniform distribution
2. **Binary search** (`bisect`) makes key lookups O(log n) even with thousands of virtual nodes
3. **100–200 virtual nodes** per server is the sweet spot for balanced distribution
4. Adding a server takes ~1/N of keys evenly from all existing servers
5. Removing a server spreads its keys evenly across remaining servers

## ⏭️ Next Up

In **Notebook 3**, we'll use consistent hashing with **real Redis instances** and see how
production systems like Redis Cluster, DynamoDB, and Cassandra implement these concepts.